# data.ipynb: This file demonstrates how the data was loaded into MongoDB

In [ ]:
from pathlib import Path
from datetime import datetime
from pymongo import UpdateOne
import pandas as pd

# Uses one MongoDB document per (site, day)
csv_path = Path("Air_Quality_Realtime.csv")
if not csv_path.exists():
    csv_path = Path("/home/teaganabritten/UVA/DataScience/ds4320/Air_Quality_Realtime.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Could not find CSV at {csv_path}")

df = pd.read_csv(csv_path)

air_db = client['DS4320']
air_collection = air_db['Project2']
required = ["AQSID", "SITENAME", "PARAMETERNAME", "DATETIME_LOCAL", "REPORTINGUNITS", "VALUE"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}. Found: {list(df.columns)}")

df["DATETIME_LOCAL"] = pd.to_datetime(df["DATETIME_LOCAL"], errors="coerce")
df = df.dropna(subset=["AQSID", "SITENAME", "DATETIME_LOCAL"]).copy()
df["day"] = df["DATETIME_LOCAL"].dt.normalize()


# Unique key enforces one doc per site per day
air_collection.create_index([("site_id", 1), ("day", 1)], unique=True, name="site_day_unique")
air_collection.create_index([("day", 1)], name="day_idx")

ops = []
for (site_id, day), group in df.groupby(["AQSID", "day"], sort=False):
    site_name = str(group["SITENAME"].iloc[0])

    readings = []
    for _, row in group.iterrows():
        readings.append(
            {
                "timestamp": row["DATETIME_LOCAL"].to_pydatetime(),
                "parameter": row.get("PARAMETERNAME"),
                "value": row.get("VALUE"),
                "units": row.get("REPORTINGUNITS"),
                "datasource": row.get("DATASOURCE"),
                "objectid": row.get("OBJECTID"),
            }
        )

    day_dt = day.to_pydatetime()
    doc = {
        "site_id": str(site_id),
        "site_name": site_name,
        "day": day_dt,
        "readings": readings,
        "row_count": len(readings),
        "source_file": csv_path.name,
        "updated_at": datetime.utcnow(),
    }

    ops.append(
        UpdateOne(
            {"site_id": str(site_id), "day": day_dt},
            {"$set": doc},
            upsert=True,
        )
    )

if not ops:
    print("No valid rows to load.")
else:
    result = air_collection.bulk_write(ops, ordered=False)
    print(f"Loaded {len(ops)} site-day documents into air_quality.site_day")
    print(f"Inserted: {result.upserted_count}, Matched: {result.matched_count}, Modified: {result.modified_count}")
